# Prompt Evaluation Exercise: Add Solution Criteria
This notebook extends the eval workflow by adding per-task solution criteria and using that criteria during model-based grading.

[Open SS07 Lesson Notes: Exercise on Prompt Evals](../s04_prompt_evaluation/ss07_exercise_on_prompt_evals/index.md)

In [5]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"

## 1. Initialize Client
Load environment variables and initialize the Anthropic client and model used for the exercise.

In [6]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None,  stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

## 2. Define Helper Functions
These helpers keep chat message construction and API calls consistent across dataset generation, prompting, and grading.

In [7]:
# Function to generate a new dataset
import json


def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
        "format": "json" or "python" or "regex",
        "solution_criteria": "Key criteria for evaluating the solution"
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)

## 3. Generate Dataset with Solution Criteria
Each generated test case now includes `task`, `format`, and `solution_criteria` so the grader has clearer expectations for correctness.

[Open SS03 Lesson Notes: Generating Test Datasets](../s04_prompt_evaluation/ss03_generating_test_datasets/index.md)

In [8]:
# Generate the dataset and write it to 'dataset.json'
dataset = generate_dataset()
with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)

## 4. Persist the Exercise Dataset
Generate and save the dataset to `dataset.json` for repeatable evaluation runs.

In [9]:
# Function to grade a test case + output using a model
def grade_by_model(test_case, output):
    eval_prompt = f"""
You are an expert AWS code reviewer. Your task is to evaluate the following AI-generated solution.

Original Task:
<task>
{test_case["task"]}
</task>

Solution to Evaluate:
<solution>
{output}
</solution>

Criteria you should use to evaluate the solution:
<criteria>
{test_case["solution_criteria"]}
</criteria>

Output Format
Provide your evaluation as a structured JSON object with the following fields, in this specific order:
- "strengths": An array of 1-3 key strengths
- "weaknesses": An array of 1-3 key areas for improvement
- "reasoning": A concise explanation of your overall assessment
- "score": A number between 1-10

Respond with JSON. Keep your response concise and direct.
Example response shape:
{{
    "strengths": string[],
    "weaknesses": string[],
    "reasoning": string,
    "score": number
}}
    """

    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")
    eval_text = chat(messages, stop_sequences=["```"])
    return json.loads(eval_text)

## 5. Add Criteria-Aware Model Grading
This grader includes `solution_criteria` in the evaluation prompt so scoring is based on explicit expectations for each task.

[Open SS05 Lesson Notes: Model Based Grading](../s04_prompt_evaluation/ss05_model_based_grading/index.md)
[Open SS07 Lesson Notes: Exercise on Prompt Evals](../s04_prompt_evaluation/ss07_exercise_on_prompt_evals/index.md)

In [10]:
# Passes a test case into Claude
def run_prompt(test_case):
    prompt = f"""
Please solve the following task:

{test_case["task"]}

* Respond only with Python, JSON, or a plain Regex
* Do not add any comments or commentary or explanation
"""

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```code")
    output = chat(messages, stop_sequences=["```"])
    return output

### run_prompt
This function sends each task to Claude with strict output instructions so responses are easier to grade automatically.

In [11]:
# Functions to validate the output structure
import re
import ast


def validate_json(text):
    try:
        json.loads(text.strip())
        return 10
    except json.JSONDecodeError:
        return 0


def validate_python(text):
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0


def validate_regex(text):
    try:
        re.compile(text.strip())
        return 10
    except re.error:
        return 0


def grade_syntax(response, test_case):
    format = test_case["format"]
    if format == "json":
        return validate_json(response)
    elif format == "python":
        return validate_python(response)
    else:
        return validate_regex(response)


### Code-Based Syntax Grading
These validators score whether output parses as JSON, Python, or regex, and `grade_syntax` routes by `test_case["format"]`.

[Open SS06 Lesson Notes: Code Based Grading](../s04_prompt_evaluation/ss06_code_based_grading/index.md)

In [12]:
# Function to execute a single test case and grade the output
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)

    model_grade = grade_by_model(test_case, output)
    model_score = model_grade["score"]
    reasoning = model_grade["reasoning"]

    syntax_score = grade_syntax(output, test_case)

    score = (model_score + syntax_score) / 2

    return {
        "output": output,
        "test_case": test_case,
        "score": score,
        "reasoning": reasoning,
    }

### run_test_case
This function combines model-based grading and syntax grading, then returns output, score, and reasoning for one task.

In [13]:
from statistics import mean


def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []

    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    average_score = mean([result["score"] for result in results])
    print(f"Average score: {average_score}")

    return results

### run_eval
This function executes all test cases, collects results, and prints the average score for quick prompt iteration tracking.

## 6. Run the Evaluation
Load the saved dataset and run the full evaluation pipeline across all generated test cases.

In [14]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

Average score: 8.5


## 7. Review Results
Print formatted JSON to inspect model outputs, scores, and grader reasoning for each case.

In [15]:
print(json.dumps(results, indent=2))

[
  {
    "output": "\nimport json\nimport sys\n\ndef extract_s3_buckets(template):\n    try:\n        if isinstance(template, str):\n            template = json.loads(template)\n        \n        s3_buckets = []\n        resources = template.get('Resources', {})\n        \n        for logical_id, resource in resources.items():\n            if resource.get('Type') == 'AWS::S3::Bucket':\n                s3_buckets.append(logical_id)\n        \n        return s3_buckets\n    except json.JSONDecodeError:\n        return []\n\nif __name__ == \"__main__\":\n    template_input = sys.stdin.read()\n    result = extract_s3_buckets(template_input)\n    print(json.dumps(result))\n",
    "test_case": {
      "task": "Parse an AWS CloudFormation template and extract all resource logical IDs that have type 'AWS::S3::Bucket'",
      "format": "python",
      "solution_criteria": "Function should accept a CloudFormation JSON/dict as input and return a list of logical IDs for S3 buckets. Should handle 